# SRBD-MPC for Quadrupedal Walking — Student Version

This tutorial walks you through implementing a **convex model-predictive controller** for a quadrupedal robot, modelled as a **single rigid body** (SRBD).
The formulation is the one introduced by Di Carlo et al., *"Dynamic Locomotion in the MIT Cheetah 3 Through Convex Model-Predictive Control"* (https://doi.org/10.1109/IROS.2018.8594448).
As discussed in the lecture, they formulate the MPC as a Convex-QP, by using an external gait scheduling and footstep planning policy.
The MPC itself treats the Quadruped as a "block" and plans the forces that have to act at the respective preplanned footholds to stabilize the "block" along the desired trajectory.

To keep the exercise focused on the MPC itself, we simulate the quadruped as a **single floating rigid body** (no legs, no joints).
A gait sequencer and footstep planner are provided for you. At every control step the MPC decides the 3-D ground reaction force (GRF) that each of the four "feet" applies to the block; a Drake-based simulator integrates those forces and advances the block.

The main frames and conventions are:

- World frame: $x$ forward, $y$ left, $z$ up; all positions, velocities, foot locations, and forces in the MPC are expressed in this frame.
- Body frame: fixed to the block; the inertia matrix $I_B$ and hip offsets are specified in this frame.
- Leg order: `FR`, `FL`, `HR`, `HL` = front-right, front-left, hind-right, hind-left.
- CoM means centre of mass. GRF means ground reaction force.

## What you will implement (3 TODOs)

1. **The linearised SRBD dynamics** as a CasADi model (`build_srbd_model`).
2. **The acados OCP** (cost + friction-cone constraints) (`build_ocp`).
3. **The MPC step** (update parameters, swing/stance bounds, references; solve) (`mpc_step`).

## What is already provided (in `helper.py`)

- `RobotParams` — mass / inertia / friction / hip offsets etc.
- `GaitSequencer` — periodic trot/walk/pace contact schedules.
- `FootstepPlanner` — Raibert heuristic predicting future footholds.
- `SRBDSimulator` — Drake simulator of the block receiving world-frame GRFs.
- `reference_trajectory` — constant-velocity reference generator.
- plotting helpers (`plot_contact_schedule`, `plot_run`, ...).


## 1 — Install acados and Drake

Run this cell once; it installs `acados` (building from source under `~/acados`) and `drake` (PyPI wheel), plus their Python dependencies.


In [ ]:
from helper import install_acados, install_drake
install_acados()
install_drake()


## 2 — Imports


In [ ]:
import numpy as np
import casadi as ca
import os
import ctypes
import sys
from ctypes import RTLD_GLOBAL
import matplotlib.pyplot as plt
from scipy.linalg import block_diag
import time

acados_dir = os.path.expanduser("~/acados")
lib_dir = os.path.join(acados_dir, "lib")
template_dir = os.path.join(acados_dir, "interfaces", "acados_template")

# make python package visible before importing acados_template
if template_dir not in sys.path:
    sys.path.insert(0, template_dir)

print("Using lib dir:", lib_dir)
print("Exists:", os.path.exists(lib_dir))
print("Contents:", os.listdir(lib_dir)[:20])
# preload shared libraries explicitly
for lib in [
    "libblasfeo.so",
    "libhpipm.so",
    "libacados.so",
]:
    path = os.path.join(lib_dir, lib)
    if os.path.exists(path):
        print("Loading", path)
        ctypes.CDLL(path, mode=RTLD_GLOBAL)
    else:
        print("Not found:", path)

os.environ["ACADOS_SOURCE_DIR"] = acados_dir

from acados_template import AcadosOcp, AcadosOcpSolver, AcadosModel
print("acados_template import works")
from helper import (
    RobotParams, GaitSequencer, FootstepPlanner, SRBDSimulator,
    reference_trajectory, initial_feet, start_meshcat,
    plot_contact_schedule, plot_run, plot_feet_3d, plot_footstep_plan_2d,
    skew, Rz, LEG_NAMES,
)


## 3 — Robot parameters

These match a small quadruped (~12 kg). Feel free to change mass/inertia/friction and see how the controller copes.


In [ ]:
params = RobotParams()
print(f"mass       = {params.mass} kg")
print(f"inertia    = diag({np.diag(params.inertia)}) kg m^2")
print(f"mu         = {params.mu}")
print(f"f_z bounds = [{params.f_min}, {params.f_max}] N  (per foot)")
print(f"hip offsets (body frame):")
for i, name in enumerate(LEG_NAMES):
    print(f"  {name}: {params.hip_offsets[i]}")
print(f"nominal CoM height = {params.nominal_height} m")


## 4 — Gait sequencer

A *gait* is a periodic contact pattern. `GaitSequencer` returns the per-leg stance/swing state as a function of time.

- `trot`: diagonal pairs FR+HL and FL+HR alternate.
- `walk`: one foot in swing at a time (statically stable).
- `pace`, `bound`: lateral / front-back pairs alternate.

Pick a gait and visualise the contact schedule over a 4 s window.


In [ ]:
gait = GaitSequencer("trot")
plot_contact_schedule(gait, t0=0.0, dt=0.02, N=200)
plt.show()

## 5 — Footstep planner

The planner uses the **Raibert heuristic** to pick, for each upcoming touchdown, where the foot should land:

$$
\mathbf{p}_{\text{foot}} \;=\; \mathbf{p}_{\text{hip,TD}} \;+\; \tfrac{1}{2}\,\mathbf{v}_{\text{body,TD}}\,T_{\text{stance}} \;+\; k\,(\mathbf{v}_{\text{body}} - \mathbf{v}_{\text{cmd}}).
$$

Symbols in this heuristic:

- $\mathbf{p}_{\text{foot}}$: touchdown foot position in the world frame.
- $\mathbf{p}_{\text{hip,TD}}$: predicted hip position at touchdown.
- $\mathbf{v}_{\text{body,TD}}$: predicted body/CoM velocity at touchdown.
- $T_{\text{stance}}$: duration of the stance phase for the selected gait.
- $\mathbf{v}_{\text{cmd}}$: commanded walking velocity.
- $k$: small feedback gain that moves the next step to reduce velocity error.

Given a reference trajectory of the block, the planner produces a sequence of $(4, 3)$ **absolute world foot positions** over the MPC horizon. A stance foot is frozen at its planted world position; a swing foot is assigned its next touchdown location, although its force will be constrained to zero.

The plot below is a top-down view of one horizon sampled at the MPC grid spacing `dt_mpc`. Black dots are the predicted CoM samples. Colored dots show only the footholds that are in stance at each sample, i.e. the feet that can apply force at that MPC stage.


In [ ]:
N_horizon = 20
dt_mpc = 0.05
v_cmd = np.array([.2, 0.0, 0.0])       # commanded linear velocity [m/s]
yaw_cmd = 0.0

x0 = np.zeros(12)
x0[3:6] = [0.0, 0.0, params.nominal_height]

ref = reference_trajectory(p0=x0[3:6], yaw0=yaw_cmd, v_cmd=v_cmd,
                           omega_cmd=0.0, z_ref=params.nominal_height,
                           t0=0.0, dt=dt_mpc, N=N_horizon)

feet0 = initial_feet(params, p_body=x0[3:6], yaw=yaw_cmd)
planner = FootstepPlanner(params, gait)
feet_horizon = planner.plan_horizon(feet0, ref, v_cmd,
                                    t0=0.0, dt=dt_mpc, N=N_horizon)
contact_horizon = gait.contact_schedule(t0=0.0, dt=dt_mpc, N=N_horizon)

plot_footstep_plan_2d(ref, feet_horizon, contact_horizon)
plt.show()


## 6 — SRBD dynamics:

State $\mathbf{x} \in \mathbb{R}^{12}$:

$$
\mathbf{x} = \big[\,\underbrace{\boldsymbol{\theta}}_{\text{roll, pitch, yaw}},\;
                 \underbrace{\mathbf{p}}_{\text{x, y, z}},\;
                 \underbrace{\boldsymbol{\omega}}_{\text{world ang. vel.}},\;
                 \underbrace{\mathbf{v}}_{\text{world lin. vel.}}\big]^{\!\top}.
$$

Control $\mathbf{u} \in \mathbb{R}^{12}$ — four world-frame ground reaction forces:

$$ \mathbf{u} = [\mathbf{f}_1,\,\mathbf{f}_2,\,\mathbf{f}_3,\,\mathbf{f}_4]^{\!\top},$$ with one force  $$\mathbf{f}_i = [f_{i,x}, f_{i,y}, f_{i,z}]. $$ 

The symbols used below are:

- $m$: robot mass.
- $\mathbf{g} = [0,0,-g]^\top$: gravity acceleration in the world frame.
- $I_B$: body-frame rotational inertia, from `RobotParams.inertia`.
- $I_W$: the same inertia expressed in the world frame.
- $\mathbf{r}_i$: world-frame vector from CoM to foot/contact point $i$.
- $[\mathbf{r}_i]_\times$: skew-symmetric cross-product matrix, so $[\mathbf{r}_i]_\times\mathbf{f}_i = \mathbf{r}_i \times \mathbf{f}_i$.
- $T(\boldsymbol{\theta})$: Euler-rate matrix mapping angular velocity to roll/pitch/yaw rates.

**Continuous dynamics** (exact, nonlinear):

$$
\dot{\mathbf{p}} = \mathbf{v},\qquad
m\,\dot{\mathbf{v}} = \sum_i \mathbf{f}_i + m\,\mathbf{g},\qquad
\dot{\boldsymbol{\theta}} = T(\boldsymbol{\theta})\,\boldsymbol{\omega},\qquad
\tfrac{d}{dt}\!\left(I_W \boldsymbol{\omega}\right) = \sum_i \mathbf{r}_i \times \mathbf{f}_i.
$$

**Linearisation:** Evaluate everything at a reference yaw $\psi_{\text{ref}}$ and drop small roll/pitch contributions. Then $R(\boldsymbol{\theta}) \approx R_z(\psi_{\text{ref}})$ and
$I_W(\psi_{\text{ref}}) = R_z(\psi_{\text{ref}})\,I_B\,R_z(\psi_{\text{ref}})^{\!\top}$ is a constant-per-stage matrix. Here $R_z(\psi)$ is the rotation matrix for a yaw angle $\psi$ about the world $z$ axis. The dynamics collapse to

$$
\boxed{\;
\begin{aligned}
\dot{\boldsymbol{\theta}} &\approx R_z(\psi_{\text{ref}})^{\!\top}\,\boldsymbol{\omega}, \\
\dot{\mathbf{p}}          &= \mathbf{v}, \\
\dot{\boldsymbol{\omega}} &\approx I_W(\psi_{\text{ref}})^{-1}\sum_i \big[\mathbf{r}_i\big]_\times\,\mathbf{f}_i, \\
\dot{\mathbf{v}}          &= \tfrac{1}{m}\sum_i \mathbf{f}_i + \mathbf{g}. \\
\end{aligned}\;}
$$

This is linear in $(\mathbf{x}, \mathbf{u})$ once $\psi_{\text{ref}}$ and the relative foot vectors $\mathbf{r}_i$ are fixed at each stage. We'll pass those as *parameters* to acados and update them at every MPC step.

> **Note.** The footstep planner keeps absolute world foot positions because planted feet are fixed in the world. Before sending them to the MPC, we subtract the reference CoM position so that acados receives the MIT-style CoM-to-foot vectors $\mathbf{r}_i$.


## 7 — TODO 1: SRBD dynamics (CasADi model)

Your first task is to express the linearised SRBD dynamics symbolically so that acados can autodiff and integrate them.

The symbolic variables are already declared; **fill in the four derivative expressions** (`dtheta_dt`, `dp_dt`, `domega_dt`, `dv_dt`) using the equations in the previous section. Hints:

- `Rz_yaw = ca.vertcat(ca.horzcat(c,-s,0), ca.horzcat(s,c,0), ca.horzcat(0,0,1))` with `c=ca.cos(yaw_ref)`, `s=ca.sin(yaw_ref)`.
- `I_world = Rz_yaw @ I_body @ Rz_yaw.T`; invert with `ca.solve(I_world, ca.SX.eye(3))`.
- Cross product: `ca.cross(r_i, f_i)`, where `r_i` is the CoM-to-foot vector in world coordinates.


In [ ]:
def build_srbd_model(params):
    """Linearised SRBD model for acados.

    State  x (12):  [roll, pitch, yaw, px, py, pz, wx, wy, wz, vx, vy, vz]
    Input  u (12):  [f1_xyz, f2_xyz, f3_xyz, f4_xyz]
    Param  p (13):  [yaw_ref, r1_xyz, r2_xyz, r3_xyz, r4_xyz]
                   with r_i the CoM-to-foot vector in world coordinates.
    """
    # --- symbolic variables ------------------------------------------------
    theta = ca.SX.sym("theta", 3)
    p_com = ca.SX.sym("p_com", 3)
    omega = ca.SX.sym("omega", 3)
    v     = ca.SX.sym("v", 3)
    x = ca.vertcat(theta, p_com, omega, v)

    u = ca.SX.sym("u", 12)
    f = [u[3*i : 3*i+3] for i in range(4)]

    yaw_ref = ca.SX.sym("yaw_ref")
    r = [ca.SX.sym(f"r_{i}", 3) for i in range(4)]
    p_sym = ca.vertcat(yaw_ref, *r)

    m = params.mass
    I_body = ca.SX(params.inertia)

    # =====================================================================
    # TODO 1 — Fill in the four state derivatives below.
    #
    #   1. Build Rz_yaw (3x3 rotation about world z by yaw_ref).
    #   2. I_world = Rz_yaw @ I_body @ Rz_yaw.T    (yaw-only approximation)
    #      I_world_inv = ca.solve(I_world, ca.SX.eye(3))
    #   3. Sum forces:  sum_f = f[0] + f[1] + f[2] + f[3]
    #      Sum torques: sum_tau = sum_i ca.cross(r[i], f[i])
    #   4. Derivatives:
    #        dtheta_dt = Rz_yaw.T @ omega           (small roll/pitch approx.)
    #        dp_dt     = v
    #        domega_dt = I_world_inv @ sum_tau
    #        dv_dt     = (1/m) * sum_f + [0, 0, -g]
    # =====================================================================

    # Type here: implement Rz_yaw, I_world_inv, sum_f, sum_tau,
    # and the four derivatives dtheta_dt, dp_dt, domega_dt, dv_dt.
    raise NotImplementedError("TODO 1: implement the linearised SRBD dynamics.")

    f_expl = ca.vertcat(dtheta_dt, dp_dt, domega_dt, dv_dt)

    model = AcadosModel()
    model.name = "srbd_mpc"
    model.x = x
    model.u = u
    model.p = p_sym
    model.f_expl_expr = f_expl
    return model


# Quick structural check:
_mdl = build_srbd_model(params)


## 8 — TODO 2: build the OCP (cost + friction-cone)

Next you will construct the acados optimal control problem.

Use a **LINEAR_LS** cost with tracking variable $y_k = [\mathbf{x}_k;\;\mathbf{u}_k]$ and a weight block-diag$(Q, R)$:

$$
\ell_k = \tfrac{1}{2}\,(y_k-y_{\text{ref},k})^\top W (y_k-y_{\text{ref},k}),\qquad W=\mathrm{diag}(Q,R).
$$

Here $Q$ weights state tracking errors, $R$ regularizes force usage, $W_e$ is the terminal state weight, and `Vx`/`Vu` tell acados how to construct $y_k$ from the state and input. In this notebook, $y_k$ simply stacks the full state and full input.

The input bounds and a **friction pyramid** enforce physically plausible contact forces:

$$
\forall i:\quad -\mu\,f_{i,z} \le f_{i,x} \le \mu\,f_{i,z},\qquad -\mu\,f_{i,z} \le f_{i,y} \le \mu\,f_{i,z},\qquad f_{\min} \le f_{i,z} \le f_{\max}.
$$

Symbols and acados names:

- $\mu$: coefficient of friction.
- $f_{\min}, f_{\max}$: lower/upper vertical force bounds for a stance foot.
- `lbu`, `ubu`, `idxbu`: lower/upper box bounds on selected input components.
- `C`, `D`, `lg`, `ug`: matrices/vectors for general linear inequalities $lg \le Cx + Du \le ug$.
- In this exercise `C = 0`; only `D` is needed because the friction pyramid depends on forces, not directly on the state.

**What to fill in (inside the marked TODO region):**

- `ocp.cost.*` — set `cost_type`, `W`, `Vx`, `Vu`, `yref` (stage) and their terminal counterparts.
- Tune `Q` and `R` if you like (sensible defaults are provided).
- Build the $16\times n_u$ matrix `D` for the four friction-pyramid inequalities per leg.


In [ ]:
def build_ocp(params, N, dt_mpc):
    """Assemble the acados OCP and return a compiled solver."""
    ocp = AcadosOcp()
    ocp.model = build_srbd_model(params)

    n_x = ocp.model.x.size()[0]
    n_u = ocp.model.u.size()[0]
    n_p = ocp.model.p.size()[0]
    ny = n_x + n_u

    ocp.dims.N = N
    ocp.solver_options.tf = N * dt_mpc
    ocp.parameter_values = np.zeros(n_p)

    # --- sensible default weights (you may tune them) ---------------------
    Q = np.diag([
        300.0, 300.0, 80.0,        # theta  (roll, pitch, yaw)
        200.0, 200.0, 400.0,       # p      (x, y, z)   -- z tracking matters most
         1.0,  1.0,  10.0,       # omega
        10.0, 10.0, 20.0,        # v
    ])
    R = 1e-5 * np.eye(n_u)

    # =====================================================================
    # TODO 2 — fill in the cost and the friction-pyramid inequality matrix.
    # =====================================================================

    # ---- (a) cost ----
    # Hints:
    #   ocp.cost.cost_type    = "LINEAR_LS"
    #   ocp.cost.cost_type_e  = "LINEAR_LS"
    #   ocp.cost.W            = block_diag(Q, R)
    #   ocp.cost.Vx           = np.vstack([np.eye(n_x), np.zeros((n_u, n_x))])
    #   ocp.cost.Vu           = np.vstack([np.zeros((n_x, n_u)), np.eye(n_u)])
    #   ocp.cost.yref         = np.zeros(ny)
    #   ocp.cost.W_e          = 100.0 * Q
    #   ocp.cost.Vx_e         = np.eye(n_x)
    #   ocp.cost.yref_e       = np.zeros(n_x)

    # Type here: configure the LINEAR_LS stage and terminal costs.

    # ---- (b) initial state (given) ----
    x0 = np.zeros(n_x)
    x0[5] = params.nominal_height
    ocp.constraints.x0 = x0

    # ---- (c) input box bounds (default loose; tightened per stage at runtime) ----
    lbu = np.tile([-params.mu * params.f_max, -params.mu * params.f_max, params.f_min], 4)
    ubu = np.tile([ params.mu * params.f_max,  params.mu * params.f_max, params.f_max], 4)
    ocp.constraints.lbu = lbu
    ocp.constraints.ubu = ubu
    ocp.constraints.idxbu = np.arange(n_u)

    # ---- (d) friction pyramid inequalities: lg <= C x + D u <= ug ----
    # 4 rows per foot (x-pyramid upper/lower, y-pyramid upper/lower):
    #    +f_ix - mu f_iz <= 0
    #    -f_ix - mu f_iz <= 0
    #    +f_iy - mu f_iz <= 0
    #    -f_iy - mu f_iz <= 0
    n_ineq = 16
    C = np.zeros((n_ineq, n_x))
    D = np.zeros((n_ineq, n_u))
    lg = -1e6 * np.ones(n_ineq)
    ug = np.zeros(n_ineq)

    # TODO 2b: populate D (16 rows). Example for leg i (row offsets 4*i + {0..3}):
    #   fx, fy, fz = 3*i, 3*i+1, 3*i+2
    #   D[4*i + 0, fx] =  1.0; D[4*i + 0, fz] = -params.mu
    #   D[4*i + 1, fx] = -1.0; D[4*i + 1, fz] = -params.mu
    #   D[4*i + 2, fy] =  1.0; D[4*i + 2, fz] = -params.mu
    #   D[4*i + 3, fy] = -1.0; D[4*i + 3, fz] = -params.mu

    # Type here: populate D for all four legs.

    ocp.constraints.C = C
    ocp.constraints.D = D
    ocp.constraints.lg = lg
    ocp.constraints.ug = ug

    # Keep this error until both TODO 2 cost and D-matrix sections are complete.
    raise NotImplementedError("TODO 2: configure the OCP cost and friction pyramid.")

    # ---- solver options (given) ----
    ocp.solver_options.qp_solver = "PARTIAL_CONDENSING_HPIPM"
    ocp.solver_options.hessian_approx = "GAUSS_NEWTON"
    ocp.solver_options.integrator_type = "ERK"
    ocp.solver_options.nlp_solver_type = "SQP"
    ocp.solver_options.print_level = 0
    ocp.solver_options.qp_solver_warm_start = 1

    build_tag = int(time.time())
    solver = AcadosOcpSolver(ocp, json_file=f"acados_srbd_ocp_{build_tag}.json")
    return solver, ocp


N_MPC = 20
MPC_DT = 0.025     # prediction grid spacing [s]
DT_MPC = MPC_DT    # backwards-compatible alias
solver, ocp = build_ocp(params, N=N_MPC, dt_mpc=MPC_DT)
print("acados solver built successfully.")


## 9 — TODO 3: one MPC step

Now the runtime logic. At each control step you need to:

1. **Anchor the initial state** at the current measurement (`lbx_0 = ubx_0 = x_meas`).
2. **Push the stage parameters** `[yaw_ref_k, r_1, r_2, r_3, r_4]` to every stage (0..N). Each $r_i$ must be the CoM-to-foot vector in world coordinates. Since the planner returns absolute foot positions, compute `r_i = foot_i - p_ref_k` before calling `solver.set(k, "p", ...)`.
3. **Toggle the $f_{i,z}$ bounds** per stage based on the contact schedule:
   - stance leg: $f_{i,z}\in[f_{\min}, f_{\max}]$
   - swing  leg: $f_{i,z}\in[0, 0]$ (the friction cone will then force $f_{i,x}=f_{i,y}=0$).
4. **Push the references** `yref_k = [x_ref_k; 0]`, terminal `yref_e = x_ref_N`.
5. **Solve** with `solver.solve()` and return `u0 = solver.get(0, "u")`, the first force command of the optimized horizon.


In [ ]:
def mpc_step(solver, x_meas, ref_traj, feet_horizon, contact_schedule,
             params, N, n_x=12, n_u=12):
    """Run one MPC iteration; returns (u0, status, predicted_x_traj).

    ``feet_horizon`` stores absolute world foot positions. The acados model
    expects MIT-style CoM-to-foot vectors, so each stage subtracts the
    reference CoM position before setting the parameter vector.
    """
    # =====================================================================
    # TODO 3 — fill in the body of this function.
    # =====================================================================

    # Type here:
    #   1. Fix stage-0 state bounds to x_meas.
    #   2. For k = 0..N-1, set parameters, input bounds, and yref.
    #   3. Set terminal parameters and terminal yref.
    #   4. Solve, read u0, and collect the predicted state trajectory.
    raise NotImplementedError("TODO 3: implement one MPC step.")


## 10 — Closed-loop simulation with Drake + Meshcat

Now we close the loop where:

- `MPC_DT`: spacing of the shooting nodes inside the MPC prediction horizon.
- `CONTROL_DT`: how often the controller measures the state, solves the MPC, and applies a new force command.


In [ ]:
T_SIM = 10.0        # total simulated time [s]
CONTROL_DT = 0.005  # controller update period [s]
USE_MESHCAT = True      # live block + foot markers + force arrows
WAIT_FOR_MESHCAT = True # pause before sim so you can open the browser view
FORCE_SCALE = 0.002     # metres per Newton for Meshcat force arrows
MESHCAT_HUB_HOST = "https://cloudpendulum.m2.chalmers.se:1443" 

N_STEPS = int(T_SIM / CONTROL_DT)

# Reuse one Meshcat server across reruns of this cell, so the browser URL stays stable.
meshcat = None
if USE_MESHCAT:
    if "_srbd_meshcat" not in globals():
        _srbd_meshcat = start_meshcat(hub_host=MESHCAT_HUB_HOST)
    else:
        print(f"[meshcat] reusing existing URL: {_srbd_meshcat.web_url()}")
    _srbd_meshcat.Delete()
    meshcat = _srbd_meshcat

sim = SRBDSimulator(params, dt=CONTROL_DT, use_meshcat=USE_MESHCAT,
                    force_visual=True, force_scale=FORCE_SCALE,
                    meshcat=meshcat)

# --- reset block to standing posture ----------------------------------------
p0 = np.array([0.0, 0.0, params.nominal_height])
rpy0 = np.zeros(3)
sim.reset(p=p0, rpy=rpy0)

if USE_MESHCAT and WAIT_FOR_MESHCAT:
    input("Open the Meshcat URL in your browser, then press Enter to start the simulation...")

# --- initial foot positions (directly under the hips, on the ground) --------
current_feet = initial_feet(params, p_body=p0, yaw=0.0)
last_contact = gait.contact_state(0.0)

# --- commanded motion -------------------------------------------------------
v_cmd = np.array([0.2, 0.0, 0.0])
omega_cmd = 0.0

# --- logging buffers --------------------------------------------------------
ts_log, x_log, u_log, c_log, feet_log, ref_log, status_log = [], [], [], [], [], [], []

t_now = 0.0
for step in range(N_STEPS):
    # measurement
    x_meas = sim.state_vector_12()

    # advance planted footholds at new touchdowns BEFORE planning the horizon
    contact_now = gait.contact_state(t_now)
    for i in range(4):
        if contact_now[i] and not last_contact[i]:
            # Raibert touchdown at current body state
            current_feet[i] = planner.predicted_touchdown(
                p_body=x_meas[3:6], yaw=x_meas[2],
                v_body=x_meas[9:12], v_cmd=v_cmd, leg=i)
    last_contact = contact_now

    # reference and horizon foot plan
    ref_h = reference_trajectory(
        p0=x_meas[3:6], yaw0=x_meas[2],
        v_cmd=v_cmd, omega_cmd=omega_cmd, z_ref=params.nominal_height,
        t0=t_now, dt=MPC_DT, N=N_MPC)
    feet_h = planner.plan_horizon(current_feet, ref_h, v_cmd,
                                  t0=t_now, dt=MPC_DT, N=N_MPC)
    contact_h = gait.contact_schedule(t_now, MPC_DT, N_MPC)

    # solve MPC
    out = mpc_step(solver, x_meas, ref_h, feet_h, contact_h, params, N=N_MPC)
    if isinstance(out, tuple):
        u0, status = out[0], out[1]
    else:
        u0, status = out, 0

    # integrate in Drake with the latest GRFs held constant over one control step
    forces_world = u0.reshape(4, 3)
    sim.step(forces_world, current_feet, duration=CONTROL_DT)

    # log
    ts_log.append(t_now)
    x_log.append(x_meas.copy())
    u_log.append(u0.copy())
    c_log.append(contact_now.copy())
    feet_log.append(current_feet.copy())
    ref_log.append(ref_h[0].copy())
    status_log.append(status)

    t_now += CONTROL_DT

print(f"Finished {N_STEPS} control steps  (sim time = {t_now:.2f} s)")
print(f"solver status histogram: {np.bincount(np.asarray(status_log).astype(int))}")


## 11 — Results

Time histories of the CoM, orientation, and ground reaction forces, overlaid with the reference.


In [ ]:
ts = np.array(ts_log)
xs = np.array(x_log)
us = np.array(u_log)
cs = np.array(c_log)
feet = np.array(feet_log)
ref = np.array(ref_log)

plot_run(ts, xs, us, contacts=cs, ref=ref)

plot_feet_3d(xs, feet, forces_history=us.reshape(-1, 4, 3), every=10)
plt.show()



## 12 — Going further

Things to explore once the baseline works:

- **Switch gait** — try `walk`, `pace`, `bound`. Each changes the stance/swing ratio and the attainable speed.
- **Push disturbance** — after a few seconds, call `sim.reset(v=[..])` mid-episode (or add a perturbation force) and watch the MPC recover.
- **Change the cost weights** `Q`, `R` and observe the trade-off between tracking and peak force.
- **Increase the horizon** `N_MPC` — longer horizons cost more compute but let the controller "see" upcoming touchdowns further ahead.
- **Tilt the yaw reference** (`omega_cmd != 0`) — the robot should trace a circle.
- **Tighter friction** — lower `params.mu` and watch the controller struggle with acceleration.
